# Matriz de confusion del modelo de clasificacion

Evalua `modelo_clasificacion2.keras` (11 superclases, softmax) sobre el set de test y genera la matriz de confusion.

Antes de correrlo hay que correr `cnn_damage_classification_build_dataset.ipynb`, que baja las imagenes de S3 y deja en `data_clasificacion/` el `dataset_final.csv` y el `grupos_final.json`. Los dos notebooks tienen que correrse desde la carpeta `notebooks/`. El modelo se lee de `../models/`.

Despues correr las celdas en orden, de arriba hacia abajo. La matriz se guarda en `matriz_confusion_clasificacion.png`.

In [ ]:
# Celda 1: imports y carga del modelo
# Carga modelo_clasificacion2 (multi-clase con softmax, 11 superclases) y
# la lista de clases en el orden de las salidas del modelo. El preprocess
# va dentro del modelo, asi que las imagenes se pasan en 0-255.

import os
import json

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

MODEL_PATH = "../models/modelo_clasificacion2.keras"
CLASES_PATH = "../models/modelo_clasificacion2_clases.json"
GRUPOS_MODELO_PATH = "../models/modelo_clasificacion2_grupos.json"

for ruta in (MODEL_PATH, CLASES_PATH, GRUPOS_MODELO_PATH):
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No existe {ruta}: correr este notebook desde la carpeta notebooks/")

model = tf.keras.models.load_model(MODEL_PATH)

with open(CLASES_PATH, encoding="utf-8") as f:
    FINAL_CLASSES = json.load(f)

print(f"Modelo cargado: {MODEL_PATH} | Clases: {len(FINAL_CLASSES)}")

In [ ]:
# Celda 2: configuracion
# Usa lo que genera cnn_damage_classification_build_dataset.ipynb en
# data_clasificacion/. IMG_SIZE tiene que ser el mismo del entrenamiento.

LOCAL_DATA_DIR = "data_clasificacion"
DATASET_CSV = f"{LOCAL_DATA_DIR}/dataset_final.csv"
GRUPOS_CSV_PATH = f"{LOCAL_DATA_DIR}/grupos_final.json"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
# Celda 3: verificar el dataset y armar el split de test
# Chequea que este lo que genera cnn_damage_classification_build_dataset:
#   - dataset_final.csv y grupos_final.json existen
#   - grupos_final.json es igual a los grupos con los que se entreno el modelo
#   - las imagenes del csv estan en disco
# Si algo falla corta con un error que dice que falta.
# Despues rearma el split train/val/test del entrenamiento (mismos
# train_test_split con random_state=42) y se queda con test: imagenes
# que el modelo no vio.
# Depende de: FINAL_CLASSES, GRUPOS_MODELO_PATH (celda 1), celda 2

for ruta in (DATASET_CSV, GRUPOS_CSV_PATH):
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No existe {ruta}: correr antes cnn_damage_classification_build_dataset.ipynb")

with open(GRUPOS_CSV_PATH, encoding="utf-8") as f:
    grupos_dataset = json.load(f)
with open(GRUPOS_MODELO_PATH, encoding="utf-8") as f:
    grupos_modelo = json.load(f)
if grupos_dataset != grupos_modelo:
    raise ValueError("grupos_final.json no coincide con los grupos del modelo: el dataset y el modelo usan superclases distintas")

df = pd.read_csv(DATASET_CSV)
no_estan = [r for r in df["image_path"] if not os.path.exists(r)]
if no_estan:
    raise FileNotFoundError(f"{len(no_estan)} de {len(df)} imagenes no estan en disco (ej: {no_estan[0]}). "
                            "Volver a correr cnn_damage_classification_build_dataset.ipynb")

_, resto_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
_, test_df = train_test_split(resto_df, test_size=0.50, stratify=resto_df["label"], random_state=42)
test_df = test_df.reset_index(drop=True)

print(f"Dataset OK: {len(df)} imagenes | test: {len(test_df)} imagenes\n")
print(test_df["label"].value_counts().reindex(FINAL_CLASSES).to_string())

In [ ]:
# Celda 4: predicciones sobre test
# Lee las imagenes igual que en el entrenamiento (decode_image + resize).
# Como es softmax, la clase predicha es la de mayor probabilidad.
# Depende de: model, FINAL_CLASSES (celda 1), test_df (celda 3)

def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    return tf.image.resize(img, IMG_SIZE)

test_ds = (tf.data.Dataset.from_tensor_slices(test_df["image_path"].values)
           .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(tf.data.AUTOTUNE))

test_probs = model.predict(test_ds)
y_pred = test_probs.argmax(axis=1)
y_true = test_df["label"].map({c: i for i, c in enumerate(FINAL_CLASSES)}).values

print(f"Accuracy en test: {(y_pred == y_true).mean():.4f}")

In [ ]:
# Celda 5: reporte y matriz de confusion
# Como es multi-clase (una sola clase por imagen) hay una unica matriz
# 11x11: filas = clase real, columnas = clase predicha. Se muestra en
# cantidades y normalizada por fila (la diagonal es el recall de cada clase).
# Depende de: y_true, y_pred (celda 4), FINAL_CLASSES (celda 1)

print(classification_report(y_true, y_pred, target_names=FINAL_CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(len(FINAL_CLASSES)))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=FINAL_CLASSES, yticklabels=FINAL_CLASSES, ax=axes[0])
axes[0].set_title("Cantidad de imagenes")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", cbar=False, vmin=0, vmax=1,
            xticklabels=FINAL_CLASSES, yticklabels=FINAL_CLASSES, ax=axes[1])
axes[1].set_title("Normalizada por fila (recall)")
for ax in axes:
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.tick_params(axis="x", rotation=45)
    plt.setp(ax.get_xticklabels(), ha="right")

plt.suptitle("Matriz de confusión: CNN clasificación de daños (modelo_clasificacion2, test)")
plt.tight_layout()
plt.savefig("matriz_confusion_clasificacion.png", dpi=200, bbox_inches="tight")
plt.show()

print("""
Precision: de todo lo que el modelo predijo como una clase, qué porcentaje
           realmente era esa clase (mide falsas alarmas).
Recall:    de todo lo que realmente era esa clase, qué porcentaje el modelo
           detectó correctamente (mide casos no detectados / falsos negativos).
F1-score:  balance entre precision y recall en un solo número.
Fuera de la diagonal se ve con qué clase confunde el modelo a cada una.
""")